# 10 · 命令列 CLI

把 Python 腳本從「只能在編輯器裡跑」變成「能在終端機接受參數、可重複執行的命令列工具 command-line tool」。

## 學習目標
- 理解程式如何透過命令列引數 command-line arguments 接收外部輸入，不需修改原始碼即可改變行為。
- 能用 sys.argv 讀取簡易參數，並說明它的限制。
- 能用 argparse 設計正式 CLI，包含位置引數、選用旗標 flag、型別轉換、預設值與自動產生的說明 help。
- 理解 if __name__ == "__main__" 慣例的意義，正確區分「被執行」與「被匯入」。
- 認識 subprocess 呼叫外部程式 external program 的基本概念與使用時機。
- 能把一支腳本設計成介面清楚、可重複執行、可被他人使用的工具。

## 為什麼需要命令列介面

命令列介面 command-line interface（CLI）是讓程式在終端機 terminal 用文字指令啟動，並透過命令列引數 command-line arguments 接收外部輸入的方式。

為什麼要學：若把參數寫死 hard-code 在程式碼裡，每換一組輸入就得改原始碼、存檔、重跑，無法給別人重複使用。CLI 讓「同一支程式」靠外部輸入改變行為。

在終端機執行腳本的形狀（不執行，僅示意）：

```
python 腳本.py 引數1 引數2 --旗標 值
```

In [ ]:
# 概念：寫死參數 vs 從外部接收參數的差異對比

# 寫死名字：要換人就得改這一行原始碼
def greet_hardcoded():
    name = "Alice"                       # 名字被寫死，無法從外部更換
    return f"Hello, {name}!"

# 從外部接收名字：同一支函式可服務任何輸入
def greet(name):
    return f"Hello, {name}!"             # 名字由呼叫端決定，不需改原始碼

print(greet_hardcoded())
# 模擬命令列傳入不同名字（之後會用 sys.argv / argparse 真正從終端機取得）
for external_name in ["Alice", "Bob", "Cathy"]:
    print(greet(external_name))

## sys.argv 簡易參數讀取

sys.argv 是一個串列 list，存放啟動腳本時在終端機輸入的所有文字。它是最底層、最直接的取得引數方式，適合理解原理。

重點提醒：
- 第 0 個元素 `sys.argv[0]` 是腳本名稱，真正的引數從索引 1 開始。
- 所有引數都是字串 string，數字要自己手動做型別轉換 type conversion。
- 沒有自動驗證，引數缺漏或格式錯誤要自己檢查，這也凸顯為何需要更正式的工具。

In [ ]:
# 概念：模擬 sys.argv 讀取兩個數字並相加
import sys

# 注意：在 notebook 裡 sys.argv 是 kernel 啟動參數，不是我們要的。
#       這裡自造一個假的 argv 來示範終端機執行 `python add.py 3 5` 的情況。
fake_argv = ["add.py", "3", "5"]          # [0] 是腳本名，[1:] 才是引數

# 引數數量檢查：少於 3 個代表使用者沒給滿兩個數字
if len(fake_argv) < 3:
    print("用法：python add.py 數字1 數字2")
else:
    a = float(fake_argv[1])               # 字串轉數字，手動型別轉換
    b = float(fake_argv[2])               # 若這裡傳入非數字會丟出 ValueError
    print(f"{a} + {b} = {a + b}")

# 技巧：實際腳本中改用 sys.argv 取代 fake_argv 即可
print("目前 kernel 的 sys.argv[0]：", sys.argv[0])

## argparse 正式 CLI 設計

argparse 是 Python 標準函式庫的 CLI 模組，取代手動處理 sys.argv，自動完成引數解析、錯誤提示與說明文件，是 Python 標準的 CLI 作法。

核心物件與步驟：
1. 建立 ArgumentParser 物件。
2. 用 add_argument 方法宣告位置引數 positional argument。
3. 用 parse_args 解析引數；argparse 會自動產生 help 與用法 usage 訊息。

形狀（不執行，僅示意）：

```
parser = argparse.ArgumentParser()
parser.add_argument("名稱", help="說明")
args = parser.parse_args()
```

In [ ]:
# 概念：用 argparse 改寫加法腳本（兩個位置引數）
import argparse

# 技巧：在 notebook 裡可用 parse_args([...]) 傳入模擬引數，
#       實際腳本則直接呼叫 parse_args()，由終端機提供。
parser = argparse.ArgumentParser(description="把兩個數字相加")
parser.add_argument("a", type=float, help="第一個加數")   # 位置引數，順序固定
parser.add_argument("b", type=float, help="第二個加數")   # type=float 自動轉型

args = parser.parse_args(["3", "5"])      # 模擬終端機輸入 `python add.py 3 5`
print(f"{args.a} + {args.b} = {args.a + args.b}")

# 注意：argparse 會自動產生說明；下面印出 usage 行示意 -h 的效果
print(parser.format_usage().strip())

## argparse 進階：旗標、型別與預設值

選用引數 optional argument 以 `--名稱` 形式出現（即旗標 flag），可有可無；透過旗標、型別與預設值，讓同一支工具支援多種使用模式。

常用設定對照：

| 設定 | 作用 |
|---|---|
| `type=int` | 把輸入字串轉成指定型別 |
| `default=值` | 未提供時的預設值 default |
| `action="store_true"` | 布林開關，出現即為 True |
| `choices=[...]` | 限定只能從清單中選 |
| `required=True` | 設為必填的選用引數 |

In [ ]:
# 概念：文字處理小工具（大小寫旗標 + 帶預設值的重複次數）
import argparse

parser = argparse.ArgumentParser(description="文字處理小工具")
parser.add_argument("text", help="要處理的文字")              # 位置引數
parser.add_argument("--upper", action="store_true",          # 布林開關旗標
                    help="轉成大寫")
parser.add_argument("--repeat", type=int, default=1,         # 帶預設值的選用引數
                    help="重複次數（預設 1）")

# 模擬 `python tool.py hello --upper --repeat 3`
args = parser.parse_args(["hello", "--upper", "--repeat", "3"])

result = args.text.upper() if args.upper else args.text       # 旗標決定是否轉大寫
print((result + " ") * args.repeat)                           # 用預設 / 指定次數重複

# 注意：不給 --repeat 時會自動用 default=1，不會報錯
args2 = parser.parse_args(["world"])
print(args2.text, "重複", args2.repeat, "次")

## __main__ 慣例與腳本入口

`if __name__ == "__main__"` 是一個慣例，用來區分檔案是「被直接執行」還是「被當作模組 module 匯入 import」。

為什麼要學：每個檔案都有內建變數 `__name__`。被直接執行時它等於 `"__main__"`；被別的檔案匯入時它等於模組名稱。把主流程放進這個判斷裡，檔案就能既當可執行腳本、又能被匯入而不誤觸發主流程，這是可重用程式碼的基礎。

形狀（不執行，僅示意）：

```
def main():
    ...
if __name__ == "__main__":
    main()
```

In [ ]:
# 概念：__name__ 在「執行」與「匯入」時的值不同

# 在 notebook 中直接執行的儲存格，__name__ 就是 "__main__"
print("目前 __name__ =", __name__)

def main():
    print("main() 被呼叫：這是主流程，只有被直接執行才會跑")

# 注意：被匯入時 __name__ 會變成模組名（例如 'mytool'），
#       下面這行就不會觸發，避免 import 時誤跑主流程。
if __name__ == "__main__":
    main()                                # entry point：程式入口

## subprocess 呼叫外部程式

subprocess 模組讓 Python 程式啟動並協調其他外部程式 external program（即建立一個子行程 subprocess）。常用 `subprocess.run`，引數以串列傳入。

重點概念：
- 回傳碼 return code 為 0 通常代表成功，非 0 代表失敗。
- 可擷取輸出 output（標準輸出文字）做後續判斷。
- 何時用：需要呼叫系統指令或既有命令列工具，而非用 Python 重寫一遍時。
- 安全提醒：引數用串列傳入（不要把整串文字交給 shell），避免注入風險。

In [ ]:
# 概念：用 subprocess.run 呼叫外部程式並判斷是否成功
import sys
import subprocess

# 引數以串列傳入：這裡呼叫目前的 Python 直譯器印出一行字（跨平台都存在）
completed = subprocess.run(
    [sys.executable, "-c", "print('hello from subprocess')"],
    capture_output=True,                  # 擷取輸出而非直接顯示
    text=True,                            # 以文字（str）而非位元組處理輸出
)

print("回傳碼 return code：", completed.returncode)   # 0 表示成功
print("擷取到的輸出：", completed.stdout.strip())

# 技巧：用回傳碼判斷成敗，再決定後續流程
if completed.returncode == 0:
    print("外部程式執行成功")
else:
    print("外部程式執行失敗")

## 把腳本做成可重複執行的工具

工具化 tool 設計思維：整合前面所學，讓他人不需讀原始碼就能使用一支 CLI。

一支好用工具的介面檢查清單：

| 元素 | 設計原則 |
|---|---|
| 位置引數 | 放最關鍵、必要的輸入 |
| 旗標 flag | 放可選的模式切換與調整 |
| 預設值 default | 給合理預設，常見情境免設定即可用 |
| 說明 help | 每個引數都寫清楚用途 |
| 結束狀態碼 exit code | 0 成功、非 0 失敗，方便被其他流程組合 composable |

In [ ]:
# 概念：用 argparse + main + exit code 組成一支自足的小工具
import argparse

def build_parser():
    parser = argparse.ArgumentParser(description="判斷數值是否超過上限的小工具")
    parser.add_argument("value", type=float, help="要檢查的數值")
    parser.add_argument("--limit", type=float, default=100.0,   # 合理預設值
                        help="上限（預設 100）")
    return parser

def main(argv):
    args = build_parser().parse_args(argv)
    over = args.value > args.limit
    print(f"value={args.value} limit={args.limit} -> {'超標' if over else '合格'}")
    # 用結束碼回報成敗：超標回 1，合格回 0（這裡用 return 模擬，腳本中用 sys.exit）
    return 1 if over else 0

# 模擬兩種終端機呼叫，觀察結束碼差異
print("exit code：", main(["120"]))            # 用預設 limit=100，超標
print("exit code：", main(["80", "--limit", "90"]))  # 合格

## 練習

以下三題由淺入深，皆為複合型但簡短。資料自己造，不引用任何外部檔案。

In [ ]:
# TODO 1 ·（簡單）容積率快查工具（整合：sys.argv 簡易參數讀取 + 手動型別轉換）
#   情境：用命令列輸入一棟建築的樓地板面積 GFA（gross floor area）與占地面積
#         footprint area，立刻算出容積率 FAR（floor area ratio）。
#   要求：
#     1. 用 sys.argv 讀取兩個引數（GFA、footprint）；它們都是字串 string，
#        要用 float 手動做型別轉換。（notebook 中可先用 fake_argv 模擬）
#     2. 計算 FAR = GFA / footprint，並用 print 印出結果（保留兩位小數）。
#   驗收：等同在終端機執行 `python far.py 12000 3000`，應看到 `FAR = 4.00`。
# TODO: 在這裡完成

In [ ]:
# TODO 2 ·（中間）建蔽率檢核 CLI（整合：argparse 位置引數 + 旗標 + 型別轉換 + 預設值 + __main__ 慣例）
#   情境：審查一筆基地資料，輸入占地面積與建築投影面積，計算建蔽率
#         coverage（building coverage ratio），並依政策上限判定是否合格。
#   要求：
#     1. 用 argparse 設計兩個位置引數（site area 基地面積、footprint 投影面積），
#        都用 type=float，並為每個引數寫 help。
#     2. 加旗標 --limit（type=float, default=0.6）代表建蔽率上限；
#        再加布林開關 --verbose（store_true）控制是否印出詳細計算過程。
#     3. coverage = footprint / site area；超過 limit 印「超標」、否則印「合格」。
#        把主流程包進 main 函式，並用 if __name__ == "__main__" 呼叫它
#        （notebook 中可用 parse_args([...]) 模擬輸入）。
#   驗收：等同執行 `python coverage.py 1000 700 --verbose`，
#         應看到 coverage 約 0.70 且判定為「超標」。
# TODO: 在這裡完成

In [ ]:
# TODO 3 ·（稍難）街廓樓高聚合分析器（整合：argparse 旗標與 choices + numpy 聚合 + 情境前後比較 + exit code）
#   情境：一個街廓 block 內多棟建築的樓高 height（公尺）資料，做聚合統計，
#         並比較套用「高度乘數」政策情境前後的差異。
#   要求：
#     1. 在程式內用 numpy 自造一組仿真樓高資料，例如
#        heights = np.array([12.0, 30.0, 8.0, 45.0, 21.0, 60.0])（不可引用外部檔案）。
#     2. 用 argparse 加 --stat 旗標搭配 choices（限定 mean/max/min）決定聚合統計量；
#        再加 --multiplier（type=float, default=1.0）模擬政策放寬後的高度乘數。
#     3. 對「原始樓高」與「乘上 multiplier 後的樓高」各算一次選定統計量，
#        印出兩者與差值；若放寬後的統計量超過 50 公尺，用 sys.exit(1) 回報，
#        否則以 0 結束，讓此工具可被其他流程判斷成敗。
#   驗收：等同執行 `python block.py --stat max --multiplier 1.2`，
#         應看到原始 max、放寬後 max 與差值，且因放寬後超過 50 公尺而以 exit code 1 結束。
# TODO: 在這裡完成

## 小結

- 命令列引數 command-line arguments 讓同一支程式靠外部輸入改變行為，不必修改原始碼。
- sys.argv 是最底層的取得方式：`[0]` 是腳本名、引數皆為字串、需手動型別轉換與檢查。
- argparse 是標準作法：位置引數、選用旗標 flag、type 轉型、default 預設值、choices 限定、store_true 開關，並自動產生 help 與 usage。
- `if __name__ == "__main__"` 區分「被執行」與「被匯入」，把主流程放進 main 函式是可重用程式碼的基礎。
- subprocess 用串列傳入引數呼叫外部程式，並以回傳碼 return code 判斷成敗。
- 好用工具的特質：清楚的介面與 help、合理的預設值、用結束狀態碼 exit code 表達成敗，方便被其他流程組合 composable。